# ONNX Graph Optimizations - Full Pipeline Demo

This notebook demonstrates the complete ONNX optimization workflow:

1. **Model Creation** - Define a simple CNN in PyTorch
2. **ONNX Export** - Convert to ONNX format with shape validation
3. **Graph Analysis** - Inspect nodes, shapes, and operator distribution
4. **Optimization** - Apply ORT optimization levels and measure node reduction
5. **Benchmarking** - Compare latency and throughput across models
6. **Visualization** - Charts showing optimization impact

---
**Prerequisites**: Run from the project root directory with `config.yaml` present.

```
cd projects/onnx_graph_optimizations
jupyter notebook src/notebooks/onnx_optimization_demo.ipynb
```

In [ ]:
# ── Setup: ensure project root is on the path ──────────────────────────────
import sys
import os
from pathlib import Path

# Navigate to project root (two levels up from this notebook)
notebook_dir = Path().resolve()
project_root = notebook_dir.parents[1]   # onnx_graph_optimizations/
sys.path.insert(0, str(project_root))

CONFIG_PATH = project_root / 'config.yaml'
print(f'Project root : {project_root}')
print(f'Config path  : {CONFIG_PATH}')
assert CONFIG_PATH.exists(), f'config.yaml not found at {CONFIG_PATH}'

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────────────
import yaml
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import warnings
warnings.filterwarnings('ignore')

# Project modules
from src.model_exporter.pytorch_exporter import PyTorchExporter, load_config
from src.graph_optimizer.optimization_pipeline import OptimizationPipeline
from src.graph_optimizer.constant_folder import ConstantFolder
from src.graph_optimizer.fusion_analyzer import FusionAnalyzer
from src.inference_engine.ort_inference import OrtInferenceSession
from src.inference_engine.benchmark import InferenceBenchmark
from src.inference_engine.execution_providers import ExecutionProviderSelector
from src.graph_analysis.graph_inspector import GraphInspector
from src.graph_analysis.shape_analyzer import ShapeAnalyzer
from src.graph_analysis.node_counter import NodeCounter

print('All imports successful')

In [ ]:
# ── Load configuration ───────────────────────────────────────────────────────
config = load_config(str(CONFIG_PATH))

print('Configuration loaded:')
print(f"  opset_version      : {config['model']['opset_version']}")
print(f"  pytorch in_channels: {config['pytorch_model']['in_channels']}")
print(f"  pytorch num_classes: {config['pytorch_model']['num_classes']}")
print(f"  optimization levels: {config['optimization']['levels']}")
print(f"  benchmark iters    : {config['inference']['benchmark_iterations']}")

---
## Step 1: Build and Export the PyTorch CNN to ONNX

In [ ]:
# ── Export PyTorch CNN → ONNX ────────────────────────────────────────────────
exporter = PyTorchExporter(config)
model_pt = exporter.build_model()
print(model_pt)

onnx_path = exporter.export(model_pt)
validated_model = exporter.validate(onnx_path)
print(f'\nExported ONNX model: {onnx_path}')

---
## Step 2: Graph Analysis - Inspect the Exported Model

In [ ]:
# ── Inspect graph structure ──────────────────────────────────────────────────
inspector = GraphInspector(config)
model_proto = inspector.load(onnx_path)
summary = inspector.inspect(model_proto)

print(f"\nGraph summary:")
print(f"  Total nodes     : {summary['node_count']}")
print(f"  Total parameters: {summary['total_parameters']:,}")
print(f"  Inputs          : {[i['name'] for i in summary['inputs']]}")
print(f"  Outputs         : {[o['name'] for o in summary['outputs']]}")

In [ ]:
# ── Print computation graph ───────────────────────────────────────────────────
inspector.print_graph_structure(model_proto)

In [ ]:
# ── Shape propagation ─────────────────────────────────────────────────────────
shape_analyzer = ShapeAnalyzer(config)
shape_analyzer.print_propagation(onnx_path)
dynamic_dims = shape_analyzer.find_dynamic_dimensions(onnx_path)
print(f'\nDynamic dimensions: {dynamic_dims}')

In [ ]:
# ── Operator distribution chart ───────────────────────────────────────────────
counter = NodeCounter(config)
baseline_counts = counter.count(onnx_path)
chart_path = counter.plot_distribution(
    baseline_counts,
    title='Baseline CNN - Operator Distribution',
    filename='baseline_operator_distribution.png'
)

img = plt.imread(str(chart_path))
fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()
print(f'\nBaseline operator counts: {baseline_counts}')

---
## Step 3: Detect Optimization Opportunities

In [ ]:
# ── Fusion pattern detection ──────────────────────────────────────────────────
fusion_analyzer = FusionAnalyzer(config)
matches = fusion_analyzer.find_patterns(onnx_path)

print(f'\nFusion opportunities detected: {len(matches)}')
for m in matches:
    print(f'  [{m.start_node_index}] {m.pattern.name}  →  {m.pattern.fused_op_name}')
    print(f'       Required ORT level: {m.pattern.required_opt_level}')

In [ ]:
# ── Constant folding analysis ─────────────────────────────────────────────────
folder = ConstantFolder(config)
fold_stats = folder.analyse(onnx_path)
print('\nConstant folding stats:')
for k, v in fold_stats.items():
    print(f'  {k}: {v}')

In [ ]:
# ── Detect unfused patterns ───────────────────────────────────────────────────
opportunities = inspector.detect_optimization_opportunities(model_proto)
print(f'\nOptimization opportunities ({len(opportunities)}):')
for opp in opportunities:
    print(f'  • {opp}')

---
## Step 4: Apply ORT Optimization Levels

In [ ]:
# ── Run optimization pipeline ─────────────────────────────────────────────────
pipeline = OptimizationPipeline(config)
opt_results = pipeline.run(onnx_path)

print('\nOptimization results:')
print(f'{"Level":<30} {"Nodes":>8} {"BatchNorm":>10} {"Time(s)":>10}')
print('-' * 65)
for r in opt_results:
    bn = r.op_type_counts.get('BatchNormalization', 0)
    print(f'{r.level_name:<30} {r.node_count:>8} {bn:>10} {r.elapsed_session_init_s:>10.3f}')

In [ ]:
# ── Visualize node count reduction ────────────────────────────────────────────
level_names  = [r.level_name.replace('ORT_', '') for r in opt_results]
node_counts  = [r.node_count for r in opt_results]
bn_counts    = [r.op_type_counts.get('BatchNormalization', 0) for r in opt_results]
baseline_n   = summary['node_count']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: total node count across levels
colors = ['#c6dbef', '#6baed6', '#2171b5', '#08306b']
bars = axes[0].bar(level_names, node_counts, color=colors, edgecolor='white', width=0.5)
axes[0].axhline(baseline_n, color='red', linestyle='--', linewidth=1.2, label=f'Baseline ({baseline_n} nodes)')
for bar, val in zip(bars, node_counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2, str(val),
                 ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_ylabel('Total Node Count')
axes[0].set_title('Node Count Across Optimization Levels')
axes[0].legend()
axes[0].spines['right'].set_visible(False)
axes[0].spines['top'].set_visible(False)

# Right: BatchNorm nodes remaining (shows BN folding)
axes[1].bar(level_names, bn_counts, color='#fd8d3c', edgecolor='white', width=0.5)
for i, val in enumerate(bn_counts):
    axes[1].text(i, val + 0.05, str(val), ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[1].set_ylabel('BatchNormalization Node Count')
axes[1].set_title('BatchNorm Folding Effect\n(0 = fully folded into Conv weights)')
axes[1].spines['right'].set_visible(False)
axes[1].spines['top'].set_visible(False)

plt.suptitle('ORT Optimization Impact on CNN Graph', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.xticks(rotation=15)

chart_save = Path(config['model']['output_dir']) / 'optimization_node_reduction.png'
fig.savefig(str(chart_save), dpi=120, bbox_inches='tight')
plt.show()
print(f'Chart saved: {chart_save}')

---
## Step 5: Execution Provider Selection

In [ ]:
# ── Execution providers ───────────────────────────────────────────────────────
selector = ExecutionProviderSelector(config)
available_eps = selector.list_available()
selector.report_all_providers()

# Build a safe provider list
desired = ['CUDAExecutionProvider', 'CPUExecutionProvider']
safe_providers = selector.build_provider_list(desired)
print(f'\nSafe provider list: {safe_providers}')

In [ ]:
# ── Demo CUDA switch ──────────────────────────────────────────────────────────
m_cfg = config['pytorch_model']
dummy_input = np.random.randn(1, m_cfg['in_channels'],
                               m_cfg['input_height'], m_cfg['input_width']).astype(np.float32)
active_ep = selector.demo_cuda_switch(onnx_path, dummy_input)
print(f'\nActive EP: {active_ep}')

---
## Step 6: Inference Session and Batch Inference

In [ ]:
# ── Create ORT session and run single inference ───────────────────────────────
session = OrtInferenceSession(config)
session.load(onnx_path)

outputs = session.run({'input': dummy_input})
print(f'Single inference output shape : {outputs[0].shape}')
print(f'Predicted class index         : {outputs[0].argmax(axis=1)[0]}')

In [ ]:
# ── Batch inference ───────────────────────────────────────────────────────────
big_batch = np.random.randn(64, m_cfg['in_channels'],
                             m_cfg['input_height'], m_cfg['input_width']).astype(np.float32)
batch_out = session.run_batch(big_batch, input_name='input', batch_size=16)
print(f'\nBatch inference (64 samples, bs=16) output shape: {batch_out[0].shape}')

---
## Step 7: Benchmark - Latency and Throughput

In [ ]:
# ── Batch-size sweep on baseline ─────────────────────────────────────────────
bench = InferenceBenchmark(config)
sweep_results = bench.batch_size_sweep(onnx_path, label='baseline (FP32)')

batch_sizes_plot = [r.batch_size for r in sweep_results]
latencies_plot   = [r.mean_latency_ms for r in sweep_results]
throughputs_plot = [r.throughput_samples_per_s for r in sweep_results]

In [ ]:
# ── Compare baseline vs optimized (ORT_ENABLE_ALL) ───────────────────────────
from pathlib import Path as _Path
opt_all_path = _Path(config['model']['output_dir']) / 'optimized_ORT_ENABLE_ALL.onnx'

if opt_all_path.exists():
    baseline_r, optimized_r = bench.compare_models(onnx_path, opt_all_path, batch_size=1)
    speedup = baseline_r.mean_latency_ms / optimized_r.mean_latency_ms
    print(f'\nSpeedup (ORT_ENABLE_ALL vs baseline): {speedup:.2f}x')
else:
    print('Optimized model not found - run OptimizationPipeline first')
    speedup = 1.0
    optimized_r = None

In [ ]:
# ── Performance visualization ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Latency vs batch size
axes[0].plot(batch_sizes_plot, latencies_plot, 'o-', color='steelblue',
             linewidth=2, markersize=7, label='Mean Latency (ms)')
axes[0].fill_between(batch_sizes_plot,
                     [r.p50_latency_ms for r in sweep_results],
                     [r.p95_latency_ms for r in sweep_results],
                     alpha=0.2, color='steelblue', label='P50–P95 band')
axes[0].set_xlabel('Batch Size')
axes[0].set_ylabel('Latency (ms)')
axes[0].set_title('Inference Latency vs Batch Size')
axes[0].legend()
axes[0].spines['right'].set_visible(False)
axes[0].spines['top'].set_visible(False)

# Right: Throughput vs batch size
axes[1].plot(batch_sizes_plot, throughputs_plot, 's-', color='darkorange',
             linewidth=2, markersize=7, label='Throughput (samples/s)')
axes[1].set_xlabel('Batch Size')
axes[1].set_ylabel('Samples / second')
axes[1].set_title('Inference Throughput vs Batch Size')
axes[1].legend()
axes[1].spines['right'].set_visible(False)
axes[1].spines['top'].set_visible(False)

plt.suptitle('ONNX Runtime Inference Performance (CPU)', fontsize=13, fontweight='bold')
plt.tight_layout()
perf_chart = _Path(config['model']['output_dir']) / 'inference_performance.png'
fig.savefig(str(perf_chart), dpi=120, bbox_inches='tight')
plt.show()
print(f'Performance chart saved: {perf_chart}')

In [ ]:
# ── Before/After comparison chart ────────────────────────────────────────────
if optimized_r is not None:
    fig, ax = plt.subplots(figsize=(7, 4))
    categories = ['Baseline (FP32)', f'ORT_ENABLE_ALL\n({speedup:.2f}x speedup)']
    values = [baseline_r.mean_latency_ms, optimized_r.mean_latency_ms]
    bar_colors = ['#6baed6', '#2ca02c']
    bars = ax.bar(categories, values, color=bar_colors, edgecolor='white', width=0.4)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f} ms', ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.set_ylabel('Mean Latency (ms, batch_size=1)')
    ax.set_title('Before vs After Optimization')
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    plt.tight_layout()
    cmp_chart = _Path(config['model']['output_dir']) / 'baseline_vs_optimized.png'
    fig.savefig(str(cmp_chart), dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Comparison chart saved: {cmp_chart}')
else:
    print('Skipped comparison chart (optimized model not available)')

In [ ]:
# ── Multi-model node count comparison chart ───────────────────────────────────
output_dir = _Path(config['model']['output_dir'])
models_map = {'Baseline': onnx_path}
for level in config['optimization']['levels']:
    p = output_dir / f'optimized_{level}.onnx'
    if p.exists():
        models_map[level.replace('ORT_', '')] = p

if len(models_map) > 1:
    cmp_path = counter.compare_counts(
        models_map, chart_filename='multi_level_node_counts.png'
    )
    img = plt.imread(str(cmp_path))
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.imshow(img)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    print(f'Multi-level chart: {cmp_path}')
else:
    print('Only baseline available – run OptimizationPipeline to generate optimized models')

---
## Step 8: Fusion Analysis Summary

In [ ]:
# ── Fusion comparison: baseline vs ORT_ENABLE_ALL ────────────────────────────
if opt_all_path.exists():
    fusion_analyzer.compare_fusion(onnx_path, opt_all_path)

fusion_analyzer.report_transformer_fusions()

---
## Summary

| Step | What happened |
|------|---------------|
| Model creation | SimpleCNN: 2× (Conv→BN→ReLU) + AdaptivePool + 2× Linear |
| ONNX export | `torch.onnx.export` with dynamic batch axis, opset 17 |
| Shape inference | `onnx.shape_inference.infer_shapes` propagated all tensor shapes |
| Fusion detection | Found Conv→BN→Relu patterns → candidate for ORT fusion |
| Optimization | 4 levels: DISABLE_ALL → BASIC → EXTENDED → ALL |
| BN folding | `ORT_ENABLE_EXTENDED` absorbs BN weights into preceding Conv |
| Inference | `ort.InferenceSession.run()` on CPU EP |
| Benchmarking | Warm-up then timed loop, measuring p50/p95/p99 latencies |

**Key takeaways:**
- `ORT_ENABLE_EXTENDED` removes BatchNorm nodes by folding into Conv.
- `ORT_ENABLE_ALL` applies layout optimization on top.
- Throughput scales super-linearly with batch size up to hardware limits.
- For transformer models, use `onnxruntime.transformers.optimizer` for additional fusions.